In [ ]:
# Version 1 - For final run - JPK 2026-08-06

# Quality Check insitu data and filter stations before use

# Check 1 - Data sufficiency - 80% data coverage for the study period (2001-01-01 to 2025-12-31)
# Check 2 - Outlier check  - remove observations < 0; flag values > 99.9 percentile followed by manual check
# Check 3 - Frozen sensor - same (non-zero) values for more than 8 days - flagged and manually checked
# Check 4 - Temporal Consistency - skipped for precipitation as precipitation is highly temporally uncorrelated in consecutive days
# Check 5 - Location - Check station locations are valid
# Check 6 - Spatial Homogeneity - DMCA
# Check 7 - Station Relocation - PMFT


In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
from scipy.spatial import cKDTree
import geopandas as gpd
from shapely.geometry import Point
import geodatasets
import piecewise_regression

#read insitu data
prec_raw = pd.read_csv(r"..\Data\Insitu\Raw\prec_01_25.csv")
metadata = pd.read_csv(fr"..\Data_Temp\Insitu\Raw\station_metadata_final.csv")

# temp folder
temporary_folder = r"..\Data\Insitu\Junk"
os.makedirs(temporary_folder, exist_ok = True)

start_date = pd.Timestamp("2001-01-01")
end_date = pd.Timestamp("2025-12-31")
total_days = (end_date - start_date).days + 1

In [ ]:
# Check 1 - Data Sufficiency - 80% data coverage for the study period
# additional check for removing duplicates

prec_raw['date'] = pd.to_datetime(prec_raw['date'])

prec_dup = prec_raw[prec_raw.duplicated(subset=['date', 'index_no'], keep = False)]
prec = prec_raw[~prec_raw.index.isin(prec_dup.index)]

prec_coverage = prec.groupby('index_no').agg(
    first_date = ('date', 'min'),
    last_date = ('date', 'max'),
    total_records = ('date', 'count'),
    missing_percentage = ('rainfall_sum', lambda x: (total_days - x.count()) / total_days * 100)
).reset_index()

prec_qualify_list = list((prec_coverage[prec_coverage['missing_percentage'] <= 20])['index_no'])
prec_qc1 = prec[prec['index_no'].isin(prec_qualify_list)]

print(f"Prec: {len(prec_qc1['index_no'].unique())} stations qualify out of {len(prec['index_no'].unique())}")
print(f"Prec: Data dropped from QC1:", f"{((len(prec_raw) - len(prec_qc1))/ len(prec_raw)) * 100:.2f}", "%")

In [ ]:
# Check 2 - Outlier check  - remove observations < 0; flag values > 99.9 percentile followed by manual check

prec_error = prec_qc1[prec_qc1['rainfall_sum'] < 0]
error_combinations_np = prec_error[['index_no', 'year', 'month', 'day']].to_numpy()
error_combinations_tuple = [tuple(row) for row in error_combinations_np]

prec_qc2a = prec_qc1[~prec_qc1[['index_no', 'year', 'month', 'day']].apply(tuple, axis=1).isin(error_combinations_tuple)]

prec_extreme = prec_qc2a[prec_qc2a['rainfall_sum'] > np.percentile(prec_qc2a['rainfall_sum'].to_numpy(), 99.9)]

# checked flagged values for prec_extreme; largest value 361 mm is less than recorded national average so no flagged values were dropped
prec_qc2 = prec_qc2a

In [ ]:
# Check 3 - Frozen sensor - same (non-zero) values for more than 8 days - flagged and manually checked

prec_qc2_idx = list(prec_qc2['index_no'].unique())

frozen_stations_df = pd.DataFrame()

for stn in prec_qc2_idx:
    frozen_date_stn = {'index_no': stn}
    prec_station = prec_qc2[prec_qc2['index_no'] == stn].copy()

    roll_max = prec_station['rainfall_sum'].rolling(10).max()
    roll_min = prec_station['rainfall_sum'].rolling(10).min()

    mask = ((roll_max == roll_min) & roll_max.notna() & roll_max != 0).shift(-9, fill_value=False)

    frozen_date_stn['dates'] = prec_station[mask]['date'].tolist()
    frozen_date_stn = pd.DataFrame([frozen_date_stn])
    frozen_stations_df = pd.concat([frozen_stations_df, frozen_date_stn], ignore_index = True)

# manual check for frozen_stations_df - no stations had frozen dates
prec_qc3 = prec_qc2

In [ ]:
# Check 4 - Temporal Consistency - skipped for precipitation as precipitation is highly temporally uncorrelated in consecutive days
# Check 5 - Location - Check station locations are valid
# Check 6 - Spatial Homogeneity - DMCA

# A. for each station find all stations within 100km radius
# B. calculate the running cumulative sum of daily prec values for the station (y) vs the same for the average of the other stations (x)


def latlon_to_3d(lat_deg, lon_deg):
    """Converts arrays of latitude and longitude (degrees) to 3D Cartesian units."""
    lat_rad = np.radians(lat_deg)
    lon_rad = np.radians(lon_deg)
    
    x = np.cos(lat_rad) * np.cos(lon_rad)
    y = np.cos(lat_rad) * np.sin(lon_rad)
    z = np.sin(lat_rad)
    
    return np.column_stack((x, y, z))

def find_neighbor_stations(metadata_df, max_dist_km=100.0, max_elev_diff_m=1000.0):
    """
    Finds all neighboring station index_no's within max_dist_km and max_elev_diff_m
    for each station in metadata_df.
    
    Parameters:
    - metadata_df: DataFrame containing ['index_no', 'x' (lon), 'y' (lat), 'elev' (elevation in meters)]
    - max_dist_km: Maximum horizontal distance radius in kilometers
    - max_elev_diff_m: Maximum elevation difference in meters (default 1000 m)
    
    Returns:
    - Dict[str/int, List[str/int]]: Dictionary mapping target index_no -> list of neighbor index_no's
    """
    EARTH_RADIUS_KM = 6371.0
    max_dist_rad = max_dist_km / EARTH_RADIUS_KM
    chord_threshold = 2 * np.sin(max_dist_rad / 2)
    
    # Note: latlon_to_3d expects (lat, lon) -> (y, x)
    coords_3d = latlon_to_3d(metadata_df['y'].values, metadata_df['x'].values)
    elevations = metadata_df['elev'].values
    
    tree = cKDTree(coords_3d)
    matched_indices = tree.query_ball_point(coords_3d, r=chord_threshold)
    
    stations = metadata_df['index_no'].values
    neighbors_dict = {}
    
    for i, matches in enumerate(matched_indices):
        target_stn = stations[i]
        target_elev = elevations[i]
        # Exclude self match and apply elevation difference criterion
        neighbor_stns = [
            stations[idx] for idx in matches
            if idx != i and abs(int(elevations[idx]) - int(target_elev)) <= max_elev_diff_m
        ]
        neighbors_dict[target_stn] = neighbor_stns
        
    return neighbors_dict


# Prepare dictionary of matched stations for precipitation
prec_stations = (prec_qc3['index_no'].unique().astype(str))
prec_metadata = metadata[metadata['Index No.'].isin(prec_stations)]
prec_metadata = prec_metadata[['Index No.', 'Lon', 'Lat', 'Elev']].rename(columns = {'Index No.': 'index_no',
                                                                                    'Lon': 'x',
                                                                                    'Lat': 'y',
                                                                                    'Elev': 'elev'}).reset_index(drop=True)
prec_matched_stn = find_neighbor_stations(prec_metadata)

# Check 5 - Location - Check station locations are valid

geo_prec = gpd.GeoDataFrame(
    prec_metadata, 
    geometry=gpd.points_from_xy(prec_metadata['x'], prec_metadata['y']), 
    crs="EPSG:4326"
)
world = gpd.read_file(geodatasets.get_path("naturalearth.land"))
fig, ax1 = plt.subplots(figsize=(10, 10))

# Top Subplot - Tmax Stations
world.plot(ax = ax1, color='lightgrey', edgecolor='black')
geo_prec.plot(ax= ax1, color='red', markersize=30, marker='o', label='Prec Stations')
ax.set_title("Prec Station Locations")
ax1.set_xlim([60, 120])
ax1.set_ylim([15, 45])

plt.tight_layout()
plt.show()

In [ ]:
# Check 6 - Spatial Homogeneity - DMCA
# Visual double-mass inspection: flag sustained (~1yr) daily-bias deviations
# > 25 mm from a station's OWN long-term average bias vs its reference, mark
# them on the double-mass curve for manual review, and tally a count per station.
#
# NOTES:
# - reference_average still uses median (not mean) - one bad neighbour distorting
#   everyone else's reference is a separate problem this doesn't fix (see clustering
#   discussion) - a flagged station whose neighbours are ALSO flagged in the same
#   window is still a contamination signature worth checking by eye across plots.
# - because deviation is measured against each station's OWN whole-record average
#   bias, a constant elevation-driven offset does NOT by itself trigger a flag -
#   only a CHANGE in that offset does.

DEVIATION_THRESHOLD_MM = 50     # mm
ROLLING_WINDOW_DAYS   = 365     # ~1yr - persistence is enforced by smoothing, not a separate rule
MIN_EVENT_DAYS        = 180     # drop flagged runs shorter than this (guards against a
                                 # borderline threshold-crossing counting as a whole event)
FIXED_YLIM            = (-4, 4) # identical y-scale on every station's bias panel - this is
                                 # what actually "standardizes the scale" across stations,
                                 # since the top (cumsum) panel's natural scale differs by
                                 # station record length and can't be forced comparable

def flag_and_plot_dmca(clean_df, target_stn_idx, target_col, save_dir=None, show=True):
    cumsum_col = f'{target_col}_cumsum'
    df = clean_df.copy()
    df['daily_bias'] = df[target_col] - df['reference_average']
    baseline = df['daily_bias'].mean()

    # rolling mean must respect actual calendar days, not row count - with up to 20%
    # missing days allowed by Check 1, a plain .rolling(365) on the gappy index would
    # span more than 365 calendar days. Reindex to continuous daily first (same fix
    # already used in flag_spikes, Check 4), then map back onto the real dates.
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq='D')
    rolling_full = df['daily_bias'].reindex(full_idx).rolling(
        ROLLING_WINDOW_DAYS, min_periods=int(ROLLING_WINDOW_DAYS * 0.8), center=True
    ).mean()
    df['rolling_bias'] = rolling_full.reindex(df.index)

    df['deviation'] = df['rolling_bias'] - baseline
    df['flagged'] = df['deviation'].abs() > DEVIATION_THRESHOLD_MM

    # collapse consecutive flagged days into discrete events, measured in calendar days
    df['event_id'] = (df['flagged'] != df['flagged'].shift()).cumsum()
    events = []
    for eid, grp in df[df['flagged']].groupby('event_id'):
        span_days = (grp.index.max() - grp.index.min()).days + 1
        if span_days >= MIN_EVENT_DAYS:
            events.append({
                'index_no': target_stn_idx,
                'start_date': grp.index.min().date(),
                'end_date': grp.index.max().date(),
                'duration_days': span_days,
                'mean_deviation_mm': round(grp['deviation'].mean(), 2),
                'direction': 'wet shift' if grp['deviation'].mean() > 0 else 'dry shift',
            })

    kept_dates = pd.DatetimeIndex([])
    if events:
        kept_dates = pd.DatetimeIndex(sorted(set().union(*[
            set(pd.date_range(e['start_date'], e['end_date'])) for e in events
        ])))
    df['marked'] = df.index.isin(kept_dates)

    # single-line OLS reference on the double-mass panel - visual aid only, not a test
    slope, intercept = np.polyfit(df['ref_avg_cumsum'], df[cumsum_col], 1)
    fit_line = slope * df['ref_avg_cumsum'] + intercept

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

    # Panel 1: classic double-mass curve + straight-line reference + flagged points
    ax1.plot(df['ref_avg_cumsum'], df[cumsum_col], color='steelblue', lw=1.2, label='DMC curve')
    ax1.plot(df['ref_avg_cumsum'], fit_line, color='grey', ls='--', lw=1, label='single-line reference')
    marked = df[df['marked']]
    if len(marked):
        ax1.scatter(marked['ref_avg_cumsum'], marked[cumsum_col],
                    facecolors='none', edgecolors='red', s=25, linewidths=1.2,
                    label='flagged deviation', zorder=5)
    ax1.set_xlabel('Reference cumulative sum (mm)')
    ax1.set_ylabel('Target cumulative sum (mm)')
    ax1.set_title(f'Station {target_stn_idx} — double-mass curve ({target_col})')
    ax1.legend(loc='upper left', fontsize=8)

    # Panel 2: daily bias + rolling mean, FIXED y-scale, threshold guides, flagged spans marked
    ax2.plot(df.index, df['daily_bias'], color='lightgrey', lw=0.4, label='daily bias (raw)')
    ax2.plot(df.index, df['rolling_bias'], color='black', lw=1.2, label=f'{ROLLING_WINDOW_DAYS}d rolling mean')
    ax2.axhline(baseline, color='grey', ls=':', lw=1)
    ax2.axhline(baseline + DEVIATION_THRESHOLD_MM, color='red', ls='--', lw=0.8)
    ax2.axhline(baseline - DEVIATION_THRESHOLD_MM, color='red', ls='--', lw=0.8)
    if len(marked):
        ax2.scatter(marked.index, marked['rolling_bias'],
                    facecolors='none', edgecolors='red', s=18, linewidths=1.0, zorder=5)
    ax2.set_ylim(*FIXED_YLIM)
    ax2.set_xlabel('Date')
    ax2.set_ylabel('Target − reference (mm)')
    ax2.set_title(f'{len(events)} flagged deviation(s) ≥{DEVIATION_THRESHOLD_MM}mm, ≥{MIN_EVENT_DAYS}d')
    ax2.legend(loc='upper left', fontsize=8)

    plt.tight_layout()
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        fig.savefig(os.path.join(save_dir, f'DMCA_{target_col}_{target_stn_idx}.png'), dpi=120)
    if show:
        plt.show()
    else:
        plt.close(fig)

    return {'events': events, 'daily_bias': df['daily_bias'], 'baseline': baseline}

# Run over all tmax stations. Saves one PNG per station to temporary_folder for a
# quick flip-through review; set show=False if inline rendering for ~90+ stations
# gets slow.

dmca_visual_events = []
dmca_bias_series_prec = {}    # station -> full daily_bias series, kept for Check 7

for target_stn_idx in prec_stations:
    matched_stn_idx = prec_matched_stn.get(target_stn_idx, [])
    if not matched_stn_idx:
        continue

    target_stn_df = prec_qc3[prec_qc3['index_no'] == int(target_stn_idx)].drop(columns=['date', 'index_no'])
    target_stn_df = target_stn_df.rename(columns={'rainfall_sum': 'target_prec'})

    dmca_stn = target_stn_df.copy()
    for i in matched_stn_idx:
        matched_stn_df = prec_qc3[prec_qc3['index_no'] == int(i)][['year', 'month', 'day', 'rainfall_sum']]
        matched_stn_df = matched_stn_df.rename(columns={'rainfall_sum': f'prec_{i}'})
        dmca_stn = dmca_stn.merge(matched_stn_df, on=["year", "month", "day"], how='left')

    dmca_stn['date'] = pd.to_datetime(dmca_stn[['year', 'month', 'day']])
    dmca_stn = dmca_stn.set_index('date')

    neighbor_cols = [c for c in dmca_stn.columns if c.startswith("prec_")]
    dmca_stn['reference_average'] = dmca_stn[neighbor_cols].median(axis=1)

    clean_df = dmca_stn[['target_prec', 'reference_average']].dropna().copy()
    if len(clean_df) < ROLLING_WINDOW_DAYS:   # can't judge a ~1yr shift with less than a year of overlap
        print(f"Prec station {target_stn_idx}: skipped, only {len(clean_df)} overlapping days")
        continue

    clean_df['target_prec_cumsum'] = clean_df['target_prec'].cumsum()
    clean_df['ref_avg_cumsum'] = clean_df['reference_average'].cumsum()

#    print(f'\nDMC Plot for prec station {target_stn_idx} =============')
    result = flag_and_plot_dmca(clean_df, target_stn_idx, target_col='target_prec',
                                 save_dir=temporary_folder, show=False)
    dmca_visual_events.extend(result['events'])
    dmca_bias_series_prec[target_stn_idx] = result['daily_bias']

# no stations are flagged or skipped

In [23]:
# Check 7 - Station Relocation - PMFT

import pyhomogeneity as hg

def run_homogeneity_tests(bias_series_dict, variable_label, alpha=0.05):
    rows = []
    for stn, bias in bias_series_dict.items():
        series = bias.dropna()
        if len(series) < 30:
            continue

        try:
            pettitt = hg.pettitt_test(series.to_numpy(), alpha=alpha)
        except Exception as e:
            print(f"Pettitt test failed for {variable_label} station {stn}: {e}")
            continue

        rows.append({
            'index_no': stn,
            'variable': variable_label,
            'n_obs': len(series),
            'significant': bool(pettitt.h),
            'breakpoint': series.index[pettitt.cp].date(),
            'p_value': round(float(pettitt.p), 4),
            'bias_shift_C': round(float(pettitt.avg.mu2 - pettitt.avg.mu1), 2),
        })

    return pd.DataFrame(rows)


pmft_summary_prec = run_homogeneity_tests(dmca_bias_series_prec, 'tmax')
pmft_summary_prec

,index_no,variable,n_obs,significant,breakpoint,p_value,bias_shift_C
0,1004,tmax,8703,True,2013-10-05,0.0000,1.01
1,1007,tmax,9104,True,2018-09-21,0.0001,-1.26
2,1009,tmax,7909,True,2008-09-26,0.0094,-1.51
3,1016,tmax,8563,True,2012-07-16,0.0000,2.35
4,1022,tmax,9131,True,2017-05-26,0.0019,-0.54
5,1024,tmax,8779,False,2011-06-24,0.1994,-0.33
6,1027,tmax,8442,True,2006-10-25,0.0312,-0.39
7,1029,tmax,9120,True,2017-06-12,0.0226,-0.16
8,1030,tmax,9131,False,2008-09-08,0.1530,0.40
9,1036,tmax,8443,False,2015-04-18,0.2023,-0.02


In [24]:
# Save the cleaned insitu data
output_folder = r"..\Data\Insitu\Cleaned"
os.makedirs(output_folder, exist_ok=True)
metadata_subset = metadata[['Index No.', 'Lat', 'Lon', 'Elev']].rename(columns={'Index No.': 'index_no', 'Lat': 'lat', 'Lon': 'long', 'Elev': 'elevation'})
metadata_subset['index_no'] = metadata_subset['index_no'].astype(str)

prec_qc3['index_no'] = prec_qc3['index_no'].astype(str)
prec_qc3 = prec_qc3.merge(metadata_subset, on='index_no', how='left')
prec_qc3['index_no'] = prec_qc3['index_no'].astype(int)
prec_qc3.to_parquet(os.path.join(output_folder, fr"prec_cleaned.parquet"), index = False)

# Total removal stat
print(f"Prec: {((len(prec_raw) - len(prec_qc3))/len(prec_raw))*100} % rows removed ")

Prec: 4.018679671719738 % rows removed 


In [10]:
import pandas as pd
parquet = pd.read_parquet(r"E:\ClimateData_Precipitation\Data\Insitu\Cleaned\prec_cleaned.parquet")
unique_stations = parquet.drop_duplicates(subset=['index_no', 'lat', 'long'])

unique_stations.to_csv(r"E:\ClimateData_Precipitation\Data\Insitu\Cleaned\unique_stations.csv")